# Flood analysis by district: does the topological signature generalize?
Made by: Giovanni Guarnieri Soares

The pooled analysis (`Flood_analysis_statistics_OSM.ipynb`) found that flooded streets occupy more
central positions in the network. This notebook asks whether that signature **generalizes across
São Paulo**, using two complementary designs:

- **Design 1 — stratified central area**: the existing central-area network (`edges_centrals.csv`,
  metrics computed on the whole central graph) split by district, with a within-district test per
  district and a fixed-effects heterogeneity test.
- **Design 2 — one network per district, city-wide**: an independent street graph for every São
  Paulo district with ≥5 flood events in 2019; metrics computed *within* each district's own graph;
  per-district tests pooled by random-effects meta-analysis.

Comparing the designs is itself informative: Design 1 metrics see the whole central network as
context, Design 2 metrics only see the district. Agreement means the signature is robust to that
modeling choice.

Subgraph centrality is excluded here (its dense matrix exponential is infeasible for the largest
district graphs); the six shared metrics are degree, mean shortest path length, closeness,
betweenness, eigenvector, and PageRank.

Outputs: tables in `results/`, per-district edge data in `results/districts/`, figures in `figures/`.

In [ ]:
import os
import glob
import time
import unicodedata
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox
from shapely import wkt

from scipy.stats import mannwhitneyu, spearmanr, chi2
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

ox.settings.use_cache = True
ox.settings.cache_folder = "cache"
CRS_METRIC = "EPSG:31983"  # SIRGAS 2000 / UTM 23S
FIGDIR = "figures"; os.makedirs(FIGDIR, exist_ok=True)
RESDIR = "results"; os.makedirs(f"{RESDIR}/districts", exist_ok=True)

METRICS = ["Degree", "Mean Shortest Path Length", "Closeness", "bet", "Eigenvector", "Pagerank"]
LABELS = {"Degree": "Degree", "Mean Shortest Path Length": "Mean shortest path length",
          "Closeness": "Closeness", "bet": "Betweenness", "Eigenvector": "Eigenvector",
          "Pagerank": "PageRank"}
MIN_FLOODED_EDGES = 5  # minimum flooded streets in a district to run a within-district test

floods = pd.read_csv("floods.csv", sep=";")
flood_pts = gpd.GeoDataFrame(floods, geometry=gpd.points_from_xy(floods["LONG"], floods["LAT"]),
                             crs="EPSG:4326").to_crs(CRS_METRIC)

districts_sp = gpd.read_file("Shapes/DI2010_RMSP_CEM.shp")
districts_sp = districts_sp[districts_sp["NOM_MU"] == "SAO PAULO"].to_crs(CRS_METRIC)
districts_sp["geometry"] = districts_sp.geometry.buffer(0)  # heal any invalid rings

ev_join = gpd.sjoin(flood_pts, districts_sp[["NOME", "geometry"]], how="left", predicate="within")
events_per_district = ev_join["NOME"].value_counts()
print(f"{len(districts_sp)} districts in Sao Paulo municipality; "
      f"{ev_join['NOME'].notna().sum()} of {len(flood_pts)} flood events fall inside one of them")

In [ ]:
def slug(name):
    s = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    return s.replace(" ", "_").replace("/", "-")

def mwu_effect(fl, nf):
    """Two-sided MWU p-value + rank-biserial correlation with a 95% CI
    (normal approximation on U). r > 0 means flooded streets have HIGHER values."""
    n1, n2 = len(fl), len(nf)
    U, p = mannwhitneyu(fl, nf, alternative="two-sided")
    r = 2 * U / (n1 * n2) - 1
    se_r = 2 * np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12.0) / (n1 * n2)
    return p, r, r - 1.96 * se_r, r + 1.96 * se_r, se_r

def district_tests(df, district, design):
    """Per-metric MWU + effect size for one district's edge table (needs flood_count column)."""
    fl = df[df["flood_count"] > 0]
    nf = df[df["flood_count"] == 0]
    rows = []
    for m in METRICS:
        a, b = fl[m].dropna(), nf[m].dropna()
        if len(a) < MIN_FLOODED_EDGES or len(b) < MIN_FLOODED_EDGES:
            continue
        p, r, lo, hi, se = mwu_effect(a, b)
        rows.append({"design": design, "district": district, "metric": LABELS[m],
                     "n_edges": len(df), "n_flooded": len(fl), "p_MWU": p,
                     "rank_biserial": r, "CI_low": lo, "CI_high": hi, "se": se})
    return rows

def compute_metrics(G):
    """Line-graph (street-level) metrics, written back as edge attributes of G.
    Mean shortest path and closeness come from a single all-pairs BFS pass."""
    H = nx.line_graph(G, nx.Graph)
    N = H.number_of_nodes()
    msp, clo = {}, {}
    for src, lengths in nx.all_pairs_shortest_path_length(H):
        s = sum(lengths.values()); k = len(lengths)
        msp[src] = s / k
        clo[src] = ((k - 1) / s) * ((k - 1) / (N - 1)) if s > 0 else 0.0
    nx.set_edge_attributes(G, msp, "Mean Shortest Path Length")
    nx.set_edge_attributes(G, clo, "Closeness")
    try:
        nx.set_edge_attributes(G, nx.eigenvector_centrality_numpy(H), "Eigenvector")
    except Exception:
        nx.set_edge_attributes(G, np.nan, "Eigenvector")
    nx.set_edge_attributes(G, nx.edge_betweenness_centrality(G, weight="length"), "bet")
    nx.set_edge_attributes(G, dict(H.degree()), "Degree")
    nx.set_edge_attributes(G, nx.pagerank(H), "Pagerank")
    return G

def process_district(name, poly_metric, buffer_m):
    """Build the district's own street graph (polygon buffered by buffer_m meters),
    compute metrics, match the district's flood events (50 m, in meters), and return
    the edge table restricted to edges whose midpoint lies inside the UNBUFFERED polygon."""
    poly_wgs = gpd.GeoSeries([poly_metric.buffer(buffer_m)], crs=CRS_METRIC).to_crs("EPSG:4326").iloc[0]
    G = ox.convert.to_undirected(ox.graph_from_polygon(poly_wgs, network_type="drive"))
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    G = compute_metrics(G)

    pts = flood_pts[flood_pts.within(poly_metric)]
    fc = dict.fromkeys(G.edges, 0)
    if len(pts):
        G_proj = ox.projection.project_graph(G, to_crs=CRS_METRIC)
        ne, dist = ox.distance.nearest_edges(G_proj, pts.geometry.x, pts.geometry.y, return_dist=True)
        for e, d in zip(ne, dist):
            if d < 50:
                fc[e] += 1
    nx.set_edge_attributes(G, fc, "flood_count")

    _, edges = ox.graph_to_gdfs(G)
    edges = edges.reset_index()
    mid = edges.geometry.to_crs(CRS_METRIC).interpolate(0.5, normalized=True)
    edges = edges[mid.within(poly_metric).values]
    out = pd.DataFrame(edges[["u", "v", "key", "length", "flood_count"] + METRICS])
    out["district"] = name
    return out

## 1. Design 1 — stratify the existing central-area network by district

Metrics come from `edges_centrals.csv` and were computed on the **whole central graph**; here each
edge is assigned to a district by its midpoint and the flooded vs non-flooded test runs within each
district that has at least 5 flooded streets.

In [ ]:
central = pd.read_csv("edges_centrals.csv")
central["geometry"] = central["geometry"].apply(wkt.loads)
central = gpd.GeoDataFrame(central, geometry="geometry", crs="EPSG:4326").to_crs(CRS_METRIC)

mid = gpd.GeoDataFrame(geometry=central.geometry.interpolate(0.5, normalized=True), crs=CRS_METRIC)
j = gpd.sjoin(mid, districts_sp[["NOME", "geometry"]], how="left", predicate="within")
j = j[~j.index.duplicated()]
central["district"] = j["NOME"].fillna("OUTSIDE").values

d1_rows = []
for name, df in central.groupby("district"):
    if name == "OUTSIDE":
        continue
    d1_rows += district_tests(df, name, "design1")
design1 = pd.DataFrame(d1_rows)
design1["p_BH"] = multipletests(design1["p_MWU"], method="fdr_bh")[1]
design1.to_csv(f"{RESDIR}/design1_district_tests.csv", index=False)
print(f"Design 1: {design1['district'].nunique()} districts testable "
      f"(of {central[central['district'] != 'OUTSIDE']['district'].nunique()} touched by the central network)")
design1[design1["metric"] == "Closeness"].sort_values("rank_biserial", ascending=False).round(4)

In [ ]:
# Formal heterogeneity test: does the closeness effect differ across districts?
# District fixed effects + likelihood-ratio test of the closeness x district interaction.
# Restricted to districts with >= 3 flooded edges (perfect-separation guard for the dummies).
ok_districts = central.groupby("district").apply(lambda d: (d["flood_count"] > 0).sum() >= 3)
reg = central[central["district"].isin(ok_districts[ok_districts].index)].copy()

def z(s):
    return (s - s.mean()) / s.std()

reg_df = pd.DataFrame({
    "flooded": (reg["flood_count"] > 0).astype(int),
    "closeness_z": z(reg["Closeness"]),
    "elevation_z": z(reg["elevation"]),
    "grade_abs_z": z(reg["grade_abs"]),
    "log_length_z": z(np.log(reg["length"])),
    "lanes_z": z(reg["lanes"]),
    "district": reg["district"].values,
})
import statsmodels.formula.api as smf
base = smf.logit("flooded ~ closeness_z + elevation_z + grade_abs_z + log_length_z + lanes_z + C(district)",
                 reg_df).fit(disp=0, maxiter=500)
inter = smf.logit("flooded ~ closeness_z * C(district) + elevation_z + grade_abs_z + log_length_z + lanes_z",
                  reg_df).fit(disp=0, maxiter=500)
lr = 2 * (inter.llf - base.llf)
df_lr = int(inter.df_model - base.df_model)
print(f"Closeness OR per SD (district fixed effects): {np.exp(base.params['closeness_z']):.3f} "
      f"(p = {base.pvalues['closeness_z']:.2e})")
print(f"Heterogeneity LR test (closeness x district): LR = {lr:.1f}, df = {df_lr}, "
      f"p = {chi2.sf(lr, df_lr):.3f}")
print("p > 0.05 means no evidence the closeness effect differs across central districts.")

## 2. Design 2 — one street network per district, city-wide

Districts of São Paulo municipality with **≥5 flood events** inside their polygon are processed
independently: own OSM graph, own metrics, own flood matching. The polygon is buffered before the
download so streets at the border keep their real neighborhood (buffer size chosen empirically
below), and only edges whose midpoint falls inside the *unbuffered* district enter the analysis.

In [ ]:
MIN_EVENTS = 5
sel = events_per_district[events_per_district >= MIN_EVENTS]
skipped = events_per_district[events_per_district < MIN_EVENTS]
sel_gdf = districts_sp[districts_sp["NOME"].isin(sel.index)].copy()
sel_gdf["n_events"] = sel_gdf["NOME"].map(sel)
sel_gdf["area_km2"] = sel_gdf.geometry.area / 1e6
sel_gdf = sel_gdf.sort_values("area_km2").reset_index(drop=True)  # small first: fast checkpoints

print(f"Selected: {len(sel_gdf)} districts with >= {MIN_EVENTS} events "
      f"({int(sel_gdf['n_events'].sum())} events); skipped: {len(skipped)} districts "
      f"with < {MIN_EVENTS} events ({int(skipped.sum())} events) plus "
      f"{len(districts_sp) - len(events_per_district)} districts with none.")
sel_gdf[["NOME", "n_events", "area_km2"]].round(1)

### 2.1 Buffer sensitivity — how much buffer is enough?

Cutting a graph at the district border creates artificial dead-ends: border streets lose degree,
closeness, and betweenness. A buffer restores each in-district street's neighborhood, but global
metrics are boundary-sensitive at any finite cutoff (Gil 2017), so the buffer size should be chosen
empirically, not guessed. For three districts of different sizes we build the graph with buffers of
0/250/500/1000 m and check, against the 1000 m reference: (a) the Spearman correlation of each
metric's within-district values, and (b) the shift in the closeness rank-biserial effect size. We
adopt the smallest buffer whose metrics correlate ≥ 0.99 with the reference and whose effect size
moves ≤ 0.02.

In [ ]:
BUFFERS = [0, 250, 500, 1000]
sens_idx = [0, len(sel_gdf) // 2, int(len(sel_gdf) * 0.75)]  # small / median / large (75th pct;
# the very largest district is excluded to keep this check affordable)
sens_names = sel_gdf.loc[sens_idx, "NOME"].tolist()
print("Sensitivity districts:", sens_names)

sens_data = {}
for name in sens_names:
    poly = sel_gdf.loc[sel_gdf["NOME"] == name, "geometry"].iloc[0]
    sens_data[name] = {}
    for b in BUFFERS:
        t0 = time.time()
        sens_data[name][b] = process_district(name, poly, b)
        print(f"  {name} buffer {b:>4} m: {len(sens_data[name][b])} edges "
              f"({time.time() - t0:.0f}s)")

rows = []
for name in sens_names:
    ref = sens_data[name][BUFFERS[-1]]
    for b in BUFFERS[:-1]:
        cur = sens_data[name][b]
        merged = cur.merge(ref, on=["u", "v", "key"], suffixes=("_b", "_ref"))
        corrs = {m: spearmanr(merged[f"{m}_b"], merged[f"{m}_ref"], nan_policy="omit")[0] for m in METRICS}
        row = {"district": name, "buffer_m": b, "shared_edges": len(merged),
               "min_metric_corr": min(corrs.values()), **{f"corr_{m}": v for m, v in corrs.items()}}
        for data, tag in ((cur, "b"), (ref, "ref")):
            fl = data[data["flood_count"] > 0]["Closeness"].dropna()
            nf = data[data["flood_count"] == 0]["Closeness"].dropna()
            row[f"r_closeness_{tag}"] = (mwu_effect(fl, nf)[1]
                                         if len(fl) >= MIN_FLOODED_EDGES else np.nan)
        row["effect_shift"] = abs(row["r_closeness_b"] - row["r_closeness_ref"])
        rows.append(row)
sens = pd.DataFrame(rows)
sens.to_csv(f"{RESDIR}/buffer_sensitivity.csv", index=False)
sens[["district", "buffer_m", "shared_edges", "min_metric_corr",
      "r_closeness_b", "r_closeness_ref", "effect_shift"]].round(4)

In [ ]:
# Decision rule: smallest buffer meeting both criteria in every sensitivity district
BUFFER_M = BUFFERS[-1]  # fallback: the reference itself
for b in BUFFERS[:-1]:
    sub = sens[sens["buffer_m"] == b]
    if (sub["min_metric_corr"] >= 0.99).all() and (sub["effect_shift"].fillna(0) <= 0.02).all():
        BUFFER_M = b
        break
print(f"Adopted BUFFER_M = {BUFFER_M} m "
      f"(smallest buffer with all metric correlations >= 0.99 vs the 1000 m reference "
      f"and closeness effect-size shift <= 0.02)")

### 2.2 Main loop (checkpointed)

Each district is written to `results/districts/<NAME>.csv` as soon as it finishes; re-running this
cell (or the whole notebook) skips districts already on disk, so the loop survives interruptions.

In [ ]:
failed = []
for _, drow in sel_gdf.iterrows():
    name = drow["NOME"]
    fp = f"{RESDIR}/districts/{slug(name)}.csv"
    if os.path.exists(fp):
        continue
    t0 = time.time()
    try:
        out = process_district(name, drow["geometry"], BUFFER_M)
        out.to_csv(fp, index=False)
        print(f"{name:<22} {len(out):>6} edges, {int((out['flood_count'] > 0).sum()):>3} flooded, "
              f"{int(out['flood_count'].sum()):>3} events matched ({time.time() - t0:.0f}s)")
    except Exception as err:
        failed.append(name)
        print(f"{name:<22} FAILED: {err}")
print(f"\nDone: {len(glob.glob(f'{RESDIR}/districts/*.csv'))} district files, {len(failed)} failures {failed}")

In [ ]:
# Aggregate: per-district edge tables -> per-district tests, plus a district summary
d2_rows, summary = [], []
for fp in sorted(glob.glob(f"{RESDIR}/districts/*.csv")):
    df = pd.read_csv(fp)
    name = df["district"].iloc[0]
    d2_rows += district_tests(df, name, "design2")
    summary.append({"district": name, "n_edges": len(df),
                    "n_flooded": int((df["flood_count"] > 0).sum()),
                    "events_matched": int(df["flood_count"].sum()),
                    "events_in_polygon": int(events_per_district.get(name, 0)),
                    "mean_degree": df["Degree"].mean()})
design2 = pd.DataFrame(d2_rows)
design2["p_BH"] = multipletests(design2["p_MWU"], method="fdr_bh")[1]
design2.to_csv(f"{RESDIR}/design2_district_tests.csv", index=False)
summary = pd.DataFrame(summary).sort_values("events_matched", ascending=False)
summary.to_csv(f"{RESDIR}/design2_district_summary.csv", index=False)
match_rate = summary["events_matched"].sum() / summary["events_in_polygon"].sum()
print(f"Design 2: {summary.shape[0]} districts processed, "
      f"{design2['district'].nunique()} testable (>= {MIN_FLOODED_EDGES} flooded edges); "
      f"{summary['events_matched'].sum()} of {summary['events_in_polygon'].sum()} events matched "
      f"({match_rate:.1%})")
summary.head(15)

## 3. Meta-analysis of Design 2: the distribution of the effect across the city

In [ ]:
def forest_plot(tests, metric, fname):
    sub = tests[tests["metric"] == metric].sort_values("rank_biserial").reset_index(drop=True)
    pooled, se_p, Q, dfq, tau2, I2 = dersimonian_laird(sub["rank_biserial"].values, sub["se"].values)
    fig, ax = plt.subplots(figsize=(8, 0.32 * len(sub) + 2), layout="constrained")
    ax.errorbar(sub["rank_biserial"], np.arange(len(sub)),
                xerr=[sub["rank_biserial"] - sub["CI_low"], sub["CI_high"] - sub["rank_biserial"]],
                fmt="o", color="#333", ecolor="#999", capsize=2, ms=4)
    ax.axvline(0, color="k", lw=0.8)
    ax.axvline(pooled, color="#d62728", lw=1.2, ls="--",
               label=f"pooled = {pooled:.2f} [{pooled - 1.96 * se_p:.2f}, {pooled + 1.96 * se_p:.2f}], I2 = {I2:.0f}%")
    ax.set_yticks(np.arange(len(sub)))
    ax.set_yticklabels(sub["district"], fontsize=8)
    ax.set_xlabel(f"Rank-biserial correlation ({metric}), flooded vs non-flooded")
    ax.set_title(f"Within-district effect: {metric}")
    ax.legend(loc="lower right", fontsize=8)
    plt.savefig(f"{FIGDIR}/{fname}.png", dpi=200, bbox_inches="tight")
    plt.savefig(f"{FIGDIR}/{fname}.pdf", bbox_inches="tight")
    plt.show()
    return pooled, se_p, I2

def dersimonian_laird(effects, ses):
    """Random-effects pooling (DerSimonian-Laird). Returns pooled, SE, Q, df, tau2, I2%."""
    w = 1.0 / ses ** 2
    fixed = (w * effects).sum() / w.sum()
    Q = (w * (effects - fixed) ** 2).sum()
    dfq = len(effects) - 1
    C = w.sum() - (w ** 2).sum() / w.sum()
    tau2 = max(0.0, (Q - dfq) / C) if C > 0 else 0.0
    w2 = 1.0 / (ses ** 2 + tau2)
    pooled = (w2 * effects).sum() / w2.sum()
    se_p = np.sqrt(1.0 / w2.sum())
    I2 = max(0.0, (Q - dfq) / Q) * 100 if Q > 0 else 0.0
    return pooled, se_p, Q, dfq, tau2, I2

meta_rows = []
for metric in ["Closeness", "Mean shortest path length"]:
    pooled, se_p, I2 = forest_plot(design2, metric, f"forest_{metric.split()[0].lower()}")
    meta_rows.append({"metric": metric, "pooled_r": pooled, "se": se_p, "I2_pct": I2})

# Pool every metric + sign consistency
for metric in design2["metric"].unique():
    sub = design2[design2["metric"] == metric]
    pooled, se_p, Q, dfq, tau2, I2 = dersimonian_laird(sub["rank_biserial"].values, sub["se"].values)
    n_pos = (sub["rank_biserial"] > 0).sum()
    print(f"{metric:<28} pooled r = {pooled:+.3f} [{pooled - 1.96 * se_p:+.3f}, {pooled + 1.96 * se_p:+.3f}]  "
          f"I2 = {I2:4.0f}%  positive in {n_pos}/{len(sub)} districts")

In [ ]:
# Distribution of the closeness effect across districts
sub = design2[design2["metric"] == "Closeness"]
fig, ax = plt.subplots(figsize=(7, 4.5), layout="constrained")
ax.hist(sub["rank_biserial"], bins=15, color="#555", edgecolor="white")
ax.axvline(0, color="k", lw=0.8)
ax.axvline(sub["rank_biserial"].median(), color="#d62728", ls="--",
           label=f"median = {sub['rank_biserial'].median():.2f}")
ax.set_xlabel("Rank-biserial correlation (closeness), per district")
ax.set_ylabel("Districts")
ax.set_title("How the flooded-streets-are-more-central effect is distributed across the city")
ax.legend()
plt.savefig(f"{FIGDIR}/effect_distribution.png", dpi=200, bbox_inches="tight")
plt.savefig(f"{FIGDIR}/effect_distribution.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Choropleth: district polygons colored by the closeness effect size
eff = design2[design2["metric"] == "Closeness"].set_index("district")["rank_biserial"]
plot_gdf = districts_sp.copy()
plot_gdf["effect"] = plot_gdf["NOME"].map(eff)

fig, ax = plt.subplots(figsize=(11, 11), layout="constrained")
plot_gdf.plot(ax=ax, color="#e8e8e8", edgecolor="white", linewidth=0.4)  # untested districts
vmax = np.nanmax(np.abs(plot_gdf["effect"])) or 1
plot_gdf.dropna(subset=["effect"]).plot(column="effect", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                                        ax=ax, edgecolor="black", linewidth=0.4, legend=True,
                                        legend_kwds={"shrink": 0.6, "label": "Rank-biserial (closeness)"})
flood_pts.plot(ax=ax, color="k", markersize=1, alpha=0.3)
ax.set_title("Within-district effect size across Sao Paulo (grey: too few flooded streets to test)\n"
             "black dots: 2019 flood events")
ax.axis("off")
plt.savefig(f"{FIGDIR}/district_effect_map.png", dpi=200, bbox_inches="tight")
plt.savefig(f"{FIGDIR}/district_effect_map.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Meta-regression sketch: what explains where the effect is strong?
# District mean elevation from ~100 random points per district via OpenTopoData (1 request each).
def district_mean_elevation(poly, n=100, seed=0):
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = poly.bounds
    pts = []
    while len(pts) < n:
        cand = gpd.points_from_xy(rng.uniform(minx, maxx, n), rng.uniform(miny, maxy, n))
        pts += [p for p in cand if poly.contains(p)]
    pts = gpd.GeoSeries(pts[:n], crs=CRS_METRIC).to_crs("EPSG:4326")
    locs = "|".join(f"{p.y:.5f},{p.x:.5f}" for p in pts)
    try:
        rsp = requests.get(f"https://api.opentopodata.org/v1/aster30m?locations={locs}", timeout=30)
        vals = [r["elevation"] for r in rsp.json()["results"] if r["elevation"] is not None]
        return float(np.mean(vals)) if vals else np.nan
    except Exception as err:
        print(f"  elevation lookup failed: {err}")
        return np.nan

elev_fp = f"{RESDIR}/district_elevations.csv"
if os.path.exists(elev_fp):
    delev = pd.read_csv(elev_fp, index_col=0)["mean_elevation"]
else:
    delev = {}
    for name in eff.index:
        poly = districts_sp.loc[districts_sp["NOME"] == name, "geometry"].iloc[0]
        delev[name] = district_mean_elevation(poly)
        time.sleep(1)  # public API rate limit
    delev = pd.Series(delev, name="mean_elevation")
    delev.to_csv(elev_fp)

mr = design2[design2["metric"] == "Closeness"].set_index("district").copy()
mr["mean_elevation"] = delev
mr = mr.join(summary.set_index("district")[["events_matched"]]).dropna()
X_mr = sm.add_constant(pd.DataFrame({
    "log_events": np.log(mr["events_matched"]),
    "log_edges": np.log(mr["n_edges"]),
    "mean_elevation_100m": mr["mean_elevation"] / 100,
}))
print(sm.WLS(mr["rank_biserial"], X_mr, weights=1 / mr["se"] ** 2).fit().summary())

## 4. Design 1 vs Design 2: does the modeling choice change the answer?

Same districts, same flood events — the only difference is whether a street's metrics are computed
on the whole central network (Design 1) or on the district's own network (Design 2).

In [ ]:
both = design1.merge(design2, on=["district", "metric"], suffixes=("_d1", "_d2"))
both.to_csv(f"{RESDIR}/design_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 7), layout="constrained")
colors = {m: c for m, c in zip(both["metric"].unique(), plt.cm.tab10.colors)}
for m, sub in both.groupby("metric"):
    ax.errorbar(sub["rank_biserial_d1"], sub["rank_biserial_d2"],
                xerr=1.96 * sub["se_d1"], yerr=1.96 * sub["se_d2"],
                fmt="o", ms=5, color=colors[m], ecolor=colors[m], alpha=0.6,
                elinewidth=0.8, capsize=0, label=m)
lims = [-1, 1]
ax.plot(lims, lims, "k--", lw=0.8)
ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Effect size, Design 1 (metrics on whole central network)")
ax.set_ylabel("Effect size, Design 2 (metrics on district's own network)")
ax.set_title("Per-district effect sizes under the two designs")
ax.legend(fontsize=8)
plt.savefig(f"{FIGDIR}/design_comparison.png", dpi=200, bbox_inches="tight")
plt.savefig(f"{FIGDIR}/design_comparison.pdf", bbox_inches="tight")
plt.show()

rho, p = spearmanr(both["rank_biserial_d1"], both["rank_biserial_d2"])
agree = (np.sign(both["rank_biserial_d1"]) == np.sign(both["rank_biserial_d2"])).mean()
print(f"{len(both)} district-metric pairs shared by both designs "
      f"({both['district'].nunique()} districts)")
print(f"Spearman correlation of effect sizes across designs: rho = {rho:.2f} (p = {p:.1e})")
print(f"Sign agreement: {agree:.0%}")

## Outputs

- `results/design1_district_tests.csv`, `results/design2_district_tests.csv`,
  `results/design2_district_summary.csv`, `results/design_comparison.csv`,
  `results/buffer_sensitivity.csv`, `results/district_elevations.csv`
- `results/districts/*.csv` — per-district edge tables (metrics + flood counts)
- `figures/forest_closeness`, `figures/forest_mean`, `figures/effect_distribution`,
  `figures/district_effect_map`, `figures/design_comparison` (PNG + PDF each)